# Boltz-2 screen for human GHS-R\n\nThis notebook runs the end-to-end GHS-R ligand screening workflow.

In [1]:
!pip install -U boltz pandas matplotlib pyyaml py3Dmol

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['BOLTZ_CACHE'] = '/content/drive/MyDrive/boltz_cache'
os.makedirs(os.environ['BOLTZ_CACHE'], exist_ok=True)
print('BOLTZ_CACHE =', os.environ['BOLTZ_CACHE'])

Mounted at /content/drive
BOLTZ_CACHE = /content/drive/MyDrive/boltz_cache


In [7]:
# If this repo is on GitHub, clone it. Otherwise upload files manually to /content.\n# !git clone <your-repo-url> /content/ghsr_af3_molecule_screening\n%
!unzip -o /content/ghsr_af3_molecule_screening.zip -d /content/

Archive:  /content/ghsr_af3_molecule_screening.zip
   creating: /content/ghsr_af3_molecule_screening/
  inflating: /content/ghsr_af3_molecule_screening/.DS_Store  
  inflating: /content/__MACOSX/ghsr_af3_molecule_screening/._.DS_Store  
  inflating: /content/ghsr_af3_molecule_screening/requirements.txt  
  inflating: /content/ghsr_af3_molecule_screening/README.md  
   creating: /content/ghsr_af3_molecule_screening/results/
  inflating: /content/ghsr_af3_molecule_screening/.gitignore  
   creating: /content/ghsr_af3_molecule_screening/scripts/
   creating: /content/ghsr_af3_molecule_screening/.git/
   creating: /content/ghsr_af3_molecule_screening/data/
   creating: /content/ghsr_af3_molecule_screening/outputs/
   creating: /content/ghsr_af3_molecule_screening/notebooks/
   creating: /content/ghsr_af3_molecule_screening/inputs/
   creating: /content/ghsr_af3_molecule_screening/results/top_poses/
  inflating: /content/ghsr_af3_molecule_screening/results/.gitkeep  
  inflating: /content/g

In [14]:
import os; os.chdir('/content/ghsr_af3_molecule_screening')

In [15]:
!python scripts/build_inputs.py --library data/ligand_library.csv --target-fasta data/target_ghsr.fasta --out-dir inputs

Wrote 29 YAML files to inputs
Skipped 20 entries with missing/invalid fields


In [ ]:
!python scripts/run_screen.py --inputs-dir inputs --outputs-dir outputs --recycling-steps 3 --diffusion-samples 1 --msa-mode server

[RUN ] allicin
MSA server enabled: https://api.colabfold.com
MSA server authentication: no credentials provided
Extracting the CCD data to /content/drive/MyDrive/boltz_cache/mols. This may take a bit of time. You may change the cache directory with the --cache flag.


In [ ]:
!python scripts/parse_results.py --library data/ligand_library.csv --outputs-dir outputs --results-dir results\n!python scripts/visualize.py --ranking-csv results/ranking.csv --outputs-dir outputs --plots-dir results/plots --top-poses-dir results/top_poses

In [ ]:
import pandas as pd
df = pd.read_csv('results/ranking.csv')
df.head(20)

In [ ]:
import py3Dmol
from pathlib import Path
top_pose_dir = Path('results/top_poses')
poses = sorted(top_pose_dir.glob('*.pdb'))
print('Found', len(poses), 'pose files')
if poses:
  model = poses[0].read_text()
  view = py3Dmol.view(width=900, height=600)
  view.addModel(model, 'pdb')
  view.setStyle({'cartoon': {'color': 'spectrum'}})
  view.setStyle({'resn': ['LIG']}, {'stick': {}})
  view.zoomTo()
  view.show()